# 擎天柱网站摘要器（Optimus Prime Website Summarizer）

## 练习目标（理念）

复用 Day 1 的「抓取 + Chat Completions + Markdown 展示」流水线，但把 **system prompt 换成擎天柱人设**，用本地 **Ollama**（`llama3.2`）生成戏剧化、有气势的网站摘要。

## 和本课 Day 1 / Day 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| system / user 提示 | `SYSTEM_PROMPT` + `USER_PROMPT_PREFIX` |
| messages 组装 | `messages_for(website)` |
| 本地 OpenAI 兼容 API | `OpenAI(base_url=...11434/v1)` |
| 展示 | `display(Markdown(...))` |

## 怎么跑

1. 把 `sys.path.append(...)` 改成你本机 `week1` 路径（或把 `scraper.py` 放到可导入位置）
2. 启动 Ollama 并确保有 `llama3.2`
3. 可选：`.env` 中配置 OpenAI（本笔记本主路径用 Ollama）
4. 从上到下运行；默认摘要 `https://edwarddonner.com`


In [ ]:
# ========== 导入与路径：让本机能 import 到 week1 的 scraper ==========

# sys：操作 Python 模块搜索路径
import sys
# 将 week1 目录加入 path（请按你的机器修改此绝对路径；逻辑保持原样）
sys.path.append("/Users/meniksabeeshanthennakoon/PycharmProjects/llm_engineering/week1")

# 导入标准库 os：配合 dotenv 读环境变量
import os
# load_dotenv：从 .env 加载密钥到环境变量
from dotenv import load_dotenv
# OpenAI 客户端：既可连云端，也可指向 Ollama 兼容端点
from openai import OpenAI
# Markdown + display：在 Jupyter 里渲染摘要
from IPython.display import Markdown, display
# fetch_website_contents：Day 1 同款抓取工具
from scraper import fetch_website_contents

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)

# 默认云端客户端（本练习主路径虽用 ollama，仍保留原逻辑）
openai = OpenAI()
# 本地 Ollama：OpenAI 兼容接口；api_key 可为占位字符串
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# 本地模型名：须与 ollama pull / ollama list 一致
MODEL = "llama3.2"

# 系统提示：擎天柱人设做网站摘要（提示词保持英文，便于角色发挥）
SYSTEM_PROMPT = """
You are Optimus Prime, leader of the Autobots from Cybertron.
You analyze websites and summarize them with the wisdom and gravitas of a heroic Transformer.
Use dramatic, noble language. Reference Cybertron, Energon, and the Autobot cause where fitting.
Respond in markdown. Do not wrap the markdown in a code block.
"""

# 用户提示前缀：用「截获传输」叙事引出网页正文
USER_PROMPT_PREFIX = """
Autobot, a transmission has been intercepted from the human information network.
Analyze this data stream and report back to base with a summary worthy of the Autobot cause.
If it contains news or announcements, decode those transmissions too.

"""


def messages_for(website: str) -> list:
    """为给定网站正文构建 Chat Completions 所需的 messages 列表。"""
    return [
        # system：人设与输出格式
        {"role": "system", "content": SYSTEM_PROMPT},
        # user：前缀叙事 + 抓取到的网站正文
        {"role": "user", "content": USER_PROMPT_PREFIX + website},
    ]


def summarize(url: str) -> str:
    """抓取网站，并以擎天柱风格调用本地模型返回摘要文本。"""
    # 1) 抓取正文
    website = fetch_website_contents(url)
    # 2) 走 Ollama 兼容端点做 Chat Completions
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website),
    )
    # 3) 返回助手消息正文
    return response.choices[0].message.content


def display_summary(url: str) -> None:
    """对 URL 做摘要，并以 Markdown 渲染到笔记本输出区。"""
    display(Markdown(summarize(url)))


# 默认演示：摘要课程相关站点
display_summary("https://edwarddonner.com")
